<a href="https://colab.research.google.com/github/maneeha/KGLLM/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import networkx as nx
import numpy as np
from datetime import datetime
from typing import Any, List, Optional, Dict, Set, Tuple
from collections import defaultdict
import pdfplumber
from datasketch import MinHashLSH
import pandas as pd

from llama_index.core import (
    Document,
    PropertyGraphIndex,
    Settings,
)
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.llms import LLM
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core.llms import ChatMessage
from graspologic.partition import hierarchical_leiden

In [ ]:
from llama_index.embeddings.ollama import OllamaEmbedding

embed_model = OllamaEmbedding(
    model_name="nomic-embed-text:v1.5",
    base_url="http://localhost:11434",
    ollama_additional_kwargs={"mirostat": 0},
)

# # Initialize OpenAI
# llm = OpenAI(model="gpt-3.5-turbo")

from llama_index.llms.groq import Groq
llm = Groq(model="llama3-70b-8192", api_key="groq-api")

from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model



In [ ]:
# response = llm.complete("Explain the importance of low latency LLMs")

In [ ]:
class EnhancedDocument:
    """Enhanced document class with credibility scoring."""

    def __init__(self, doc: Document):
        self.doc = doc
        self.credibility_score = self._calculate_credibility()

    def _calculate_credibility(self) -> float:
        """Calculate credibility score based on metadata."""
        score = 1.0

        # Check for medical paper specific criteria
        if self.doc.metadata.get('document_type') == 'medical':
            if 'peer_reviewed' in self.doc.metadata:
                score *= 1.5
            if 'citation_count' in self.doc.metadata:
                score *= min(1 + (self.doc.metadata['citation_count'] / 1000), 2.0)
            if 'journal_impact_factor' in self.doc.metadata:
                score *= min(1 + (self.doc.metadata['journal_impact_factor'] / 10), 2.0)

        return score

class SemanticChunker:
    """Enhanced semantic chunking with credibility awareness."""

    def __init__(self, embed_model=None, chunk_size=512):
        self.embed_model = embed_model or OpenAIEmbedding()
        self.chunk_size = chunk_size
        self.splitter = SemanticSplitterNodeParser(
            buffer_size=10,
            breakpoint_percentile_threshold=95,
            embed_model=self.embed_model,
            include_metadata=True
        )

    def chunk_documents(self, documents: List[Document]) -> List[Document]:
        """Chunk documents while preserving semantic meaning."""
        enhanced_docs = [EnhancedDocument(doc) for doc in documents]

        # Sort documents by credibility score
        enhanced_docs.sort(key=lambda x: x.credibility_score, reverse=True)

        # Create nodes using semantic chunking
        nodes = []
        for enhanced_doc in enhanced_docs:
            doc_nodes = self.splitter.get_nodes_from_documents([enhanced_doc.doc])
            for node in doc_nodes:
                node.metadata['credibility_score'] = enhanced_doc.credibility_score
            nodes.extend(doc_nodes)

        return nodes

class EnhancedGraphRAGExtractor:
    """Enhanced graph extractor with context awareness and weighted edges."""

    def __init__(self, llm: LLM, max_paths: int = 10):
        self.llm = llm
        self.max_paths = max_paths
        self._setup_extraction_template()

    def _setup_extraction_template(self):
        """Set up the extraction template with context awareness."""
        self.extract_template = """
        Given the text and its context, identify entities and relationships.
        Consider the document's credibility score: {credibility_score}

        Text: {text}

        Extract:
        1. Entities with types and descriptions
        2. Relationships with weights (0-1) based on confidence
        3. Context-aware connections

        Format:
        Entity: name||type||description
        Relation: source||target||relation||weight||context
        """

    def extract(self, node: Document) -> Tuple[List[Dict], List[Dict]]:
        """Extract entities and relationships with weights."""
        response = self.llm.complete(
            self.extract_template.format(
                text=node.get_content(),
                credibility_score=node.metadata.get('credibility_score', 1.0)
            )
        )

        entities, relations = self._parse_response(response.text)
        return entities, relations

    def _parse_response(self, response: str) -> Tuple[List[Dict], List[Dict]]:
        """Parse LLM response into structured data."""
        entities = []
        relations = []

        # Implementation of parsing logic
        # ... (parsing code here)

        return entities, relations

class EnhancedGraphRAGStore(Neo4jPropertyGraphStore):
    """Enhanced graph store with community detection and LSH."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.lsh = MinHashLSH(threshold=0.7)
        self.community_summaries = {}

    def build_hierarchical_index(self):
        """Build hierarchical index using algorithmic clustering."""
        graph = self._create_nx_graph()

        # Perform hierarchical clustering
        clusters = hierarchical_leiden(graph)

        # Build LSH index for entity linking
        self._build_lsh_index(graph.nodes())

        return clusters

    def _build_lsh_index(self, entities):
        """Build LSH index for efficient entity linking."""
        for entity in entities:
            minhash = self._create_minhash(entity)
            self.lsh.insert(entity, minhash)

    def find_similar_entities(self, entity: str, threshold: float = 0.7) -> List[str]:
        """Find similar entities using LSH."""
        minhash = self._create_minhash(entity)
        return self.lsh.query(minhash)

    def _create_minhash(self, text: str) -> Any:
        """Create MinHash signature for text."""
        # Implementation of MinHash creation
        # ... (minhash creation code here)
        pass

class EnhancedGraphRAGQueryEngine(CustomQueryEngine):
    """Enhanced query engine with contextual and long-term retrieval."""

    def __init__(
        self,
        graph_store: EnhancedGraphRAGStore,
        llm: LLM,
        hyde_model: Optional[LLM] = None,
    ):
        self.graph_store = graph_store
        self.llm = llm
        self.hyde_model = hyde_model or llm

    def query(self, query_str: str) -> str:
        """Process query using enhanced retrieval methods."""
        # Generate hypothetical document using HyDE
        hyde_doc = self._generate_hyde_document(query_str)

        # Perform contextual retrieval
        context_results = self._contextual_retrieval(query_str, hyde_doc)

        # Perform long-range search using BFS
        bfs_results = self._bfs_search(query_str)

        # Combine and rank results
        final_results = self._combine_results(context_results, bfs_results)

        # Generate final answer
        return self._generate_answer(query_str, final_results)

    def _generate_hyde_document(self, query: str) -> str:
        """Generate hypothetical document using HyDE."""
        prompt = f"Generate a hypothetical document that would answer: {query}"
        return self.hyde_model.complete(prompt).text

    def _contextual_retrieval(self, query: str, hyde_doc: str) -> List[Any]:
        """Perform context-aware retrieval."""
        # Implementation of contextual retrieval
        # ... (retrieval code here)
        pass

    def _bfs_search(self, query: str, max_depth: int = 3) -> List[Any]:
        """Perform breadth-first search for long-range connections."""
        # Implementation of BFS search
        # ... (BFS code here)
        pass

    def _combine_results(self, context_results: List[Any], bfs_results: List[Any]) -> List[Any]:
        """Combine and rank different retrieval results."""
        # Implementation of result combination
        # ... (combination code here)
        pass

    def _generate_answer(self, query: str, results: List[Any]) -> str:
        """Generate final answer using combined results."""
        prompt = self._create_answer_prompt(query, results)
        return self.llm.complete(prompt).text

In [ ]:
import pdfplumber


def load_documents(pdf_dir):
    # Initialize an empty list for documents and metadata
    documents = []
    metadata_list = []

    # Specify the directory containing your PDF files
    # pdf_dir = './mini-data'

    # Loop through all files in the specified directory
    for filename in os.listdir(pdf_dir):
        if filename.endswith('.pdf'):
            pdf_path = os.path.join(pdf_dir, filename)

            # Get file metadata
            file_stat = os.stat(pdf_path)
            file_size = file_stat.st_size  # Size in bytes
            creation_date = datetime.fromtimestamp(file_stat.st_ctime)  # Creation date
            last_modified_date = datetime.fromtimestamp(file_stat.st_mtime)


            # Open and read the PDF file
            with pdfplumber.open(pdf_path) as pdf:
                for page_number, page in enumerate(pdf.pages, start=1):  # Start page numbering at 1

                    # Extract text from the page
                    text = page.extract_text()
                    if text:  # Check if the text was extracted
                        # Create metadata dictionary for each page
                        metadata = {
                            'file_name': filename,
                            'file_path': pdf_path,
                            'document_type': 'PDF',
                            'file_size': file_size,
                            'creation_date': creation_date.strftime('%Y-%m-%d %H:%M:%S'),
                            'last_modified_date': last_modified_date.strftime('%Y-%m-%d %H:%M:%S'),
                            'page_number': page_number,
                        }

                        # Append document and its metadata to the lists
                        documents.append(Document(text=text, metadata=metadata))

                        metadata_list.append(metadata)

    return documents

In [ ]:
graph_store = EnhancedGraphRAGStore(
        username="neo4j",
        password="ZoldzPyWJlI57coBcgSLKTtgW8lhRuKAp6WRW8aFuBE",
        url="neo4j+s://2659a900.databases.neo4j.io"
    )

In [ ]:
# Create document processor
chunker = SemanticChunker()
extractor = EnhancedGraphRAGExtractor(llm=llm)

In [ ]:
# Process documents
documents = load_documents("./data")  # Implementation needed
nodes = chunker.chunk_documents(documents)

APIConnectionError: Connection error.

In [ ]:
# Build graph index
index = PropertyGraphIndex(
        nodes=nodes,
        kg_extractors=[extractor],
        property_graph_store=graph_store,
        show_progress=True,
    )

In [ ]:
# Build hierarchical index and LSH
graph_store.build_hierarchical_index()

In [ ]:
# Create query engine
query_engine = EnhancedGraphRAGQueryEngine(
        graph_store=graph_store,
        llm=llm,
)

In [ ]:
# Example query
response = query_engine.query("What are the main findings in the medical papers?")
print(response)